# GramCast — Residual U-Net training (Kaggle GPU)

SIH26074. Trains the P10/P50/P90 residual U-Net on the already-built, already-verified training pairs (see `DEV_LOG.md` in the repo for the full verification history — every piece here was tested locally on CPU before this notebook was written).

**Before running this notebook:**
1. On your local machine, zip `data/processed/training_pairs/` (excluding `zonal_stats_*.csv`, that's a demo output, not training input) — it's `fine_rainfall.npy`, `coarse_rainfall_bicubic.npy`, `terrain_static.npz`, `split_indices.npz`, `valid_mask.npy`, `dates.npy`, about 76MB total.
2. Go to kaggle.com → Datasets → New Dataset → upload the zip → name it (e.g. `gramcast-training-pairs`) → keep it **Private** (this is your own unpublished project data) → Create.
3. In this notebook, click **Add Input** (right panel) → search for your dataset → add it. It'll mount at `/kaggle/input/gramcast-training-pairs/`.
4. **Settings → Accelerator → GPU** (T4 x2 or P100). Without this the notebook still runs but on CPU, defeating the point.
5. Update `DATA_DIR` in the config cell below if your dataset folder name differs.

In [ ]:
import os, sys, time, csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected. Settings -> Accelerator -> GPU, then restart.")

In [ ]:
# --- Config ---
DATA_DIR = Path("/kaggle/input/gramcast-training-pairs")  # update if your Kaggle Dataset name differs
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
LOG_DIR = Path("/kaggle/working/logs")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = time.strftime("gramcast_%Y%m%d_%H%M%S")
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 32
BASE_CHANNELS = 16
SEED = 42
RESUME_FROM = None  # set to a checkpoint path (e.g. CHECKPOINT_DIR / "<run>_last.pt") to resume after a disconnect

torch.manual_seed(SEED)

## Load the pre-built, pre-verified training pairs

In [ ]:
fine = np.load(DATA_DIR / "fine_rainfall.npy")
coarse_bicubic = np.load(DATA_DIR / "coarse_rainfall_bicubic.npy")
splits = np.load(DATA_DIR / "split_indices.npz")
terrain = dict(np.load(DATA_DIR / "terrain_static.npz"))
valid_mask = np.load(DATA_DIR / "valid_mask.npy")

print(f"fine: {fine.shape}, coarse_bicubic: {coarse_bicubic.shape}")
print(f"train/val/test: {len(splits['train'])}/{len(splits['val'])}/{len(splits['test'])} (expect 4636/244/702)")
assert fine.shape == (5582, 35, 50), f"Unexpected shape {fine.shape} - is this really the verified GramCast dataset?"
assert not np.isnan(fine).any() and not np.isnan(coarse_bicubic).any(), "NaN in loaded data - re-check the upload"

## Model code (inlined from `ml/model/`, verified locally — see `DEV_LOG.md`)

Kept identical to the tested local modules rather than reimplemented, to avoid any transcription drift from what was actually verified.

In [ ]:
# --- preprocessing.py ---
WORLDCOVER_CLASSES = [10, 20, 30, 40, 50, 60, 80, 90, 95]


def compute_normalization_stats(terrain, coarse_bicubic_train):
    return {
        "dem_mean": float(terrain["dem"].mean()), "dem_std": float(terrain["dem"].std()),
        "slope_mean": float(terrain["slope"].mean()), "slope_std": float(terrain["slope"].std()),
        "rain_mean": float(coarse_bicubic_train.mean()), "rain_std": float(coarse_bicubic_train.std()),
    }


def build_input_tensor(coarse_bicubic, terrain, stats):
    T, H, W = coarse_bicubic.shape
    rain_norm = (coarse_bicubic - stats["rain_mean"]) / stats["rain_std"]
    dem_norm = (terrain["dem"] - stats["dem_mean"]) / stats["dem_std"]
    slope_norm = (terrain["slope"] - stats["slope_mean"]) / stats["slope_std"]
    aspect_rad = np.deg2rad(terrain["aspect"])
    aspect_sin = np.sin(aspect_rad)
    aspect_cos = np.cos(aspect_rad)

    unknown_mask = ~np.isin(terrain["worldcover"], WORLDCOVER_CLASSES)
    if unknown_mask.any():
        unknown_classes = sorted(set(terrain["worldcover"][unknown_mask].tolist()))
        raise ValueError(f"WorldCover class(es) {unknown_classes} not in WORLDCOVER_CLASSES {WORLDCOVER_CLASSES}")

    static_channels = [dem_norm, slope_norm, aspect_sin, aspect_cos]
    for cls in WORLDCOVER_CLASSES:
        static_channels.append((terrain["worldcover"] == cls).astype("float32"))

    n_channels = 1 + len(static_channels)
    tensor = np.empty((T, n_channels, H, W), dtype="float32")
    tensor[:, 0, :, :] = rain_norm
    for i, ch in enumerate(static_channels):
        tensor[:, i + 1, :, :] = np.tile(ch, (T, 1, 1))
    return tensor

In [ ]:
# --- nearest_neighbor.py (predict only, for the mass-conservation target) ---
def nn_predict(fine_shape_grid):
    T, h, w = fine_shape_grid.shape
    ch, cw = h // 5, w // 5
    coarse = fine_shape_grid.reshape(T, ch, 5, cw, 5).mean(axis=(2, 4))
    return np.repeat(np.repeat(coarse, 5, axis=1), 5, axis=2)

In [ ]:
# --- mass_conservation.py ---
COARSEN_FACTOR = 5


class MassConservation(nn.Module):
    def forward(self, fine_pred, coarse_value):
        B, C, H, W = fine_pred.shape
        ch, cw = H // COARSEN_FACTOR, W // COARSEN_FACTOR

        fine_blocks = fine_pred.reshape(B, C, ch, COARSEN_FACTOR, cw, COARSEN_FACTOR)
        pred_block_mean = fine_blocks.mean(dim=(3, 5), keepdim=True)

        coarse_blocks = coarse_value.reshape(B, C, ch, COARSEN_FACTOR, cw, COARSEN_FACTOR)
        target_block_mean = coarse_blocks.mean(dim=(3, 5), keepdim=True)

        eps = 1e-6
        safe_pred_mean = torch.where(pred_block_mean > eps, pred_block_mean, torch.full_like(pred_block_mean, eps))
        scale = target_block_mean / safe_pred_mean

        rescaled_blocks = fine_blocks * scale
        near_zero_pred = pred_block_mean <= eps
        uniform_fallback = target_block_mean.expand_as(fine_blocks)
        rescaled_blocks = torch.where(near_zero_pred.expand_as(fine_blocks), uniform_fallback, rescaled_blocks)

        return rescaled_blocks.reshape(B, C, H, W)

In [ ]:
# --- unet.py --- P10/P50/P90 quantile heads, monotonicity guaranteed by construction (see DEV_LOG 2026-09-25)
PAD_H, PAD_W = 36, 52


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualUNet(nn.Module):
    def __init__(self, in_channels, base_channels=16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_channels * 2, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.correction_head = nn.Conv2d(base_channels, 3, 1)
        self.mass_conservation = MassConservation()

    def forward(self, x):
        B, C, H, W = x.shape
        pad_h, pad_w = PAD_H - H, PAD_W - W
        x_padded = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        e1 = self.enc1(x_padded)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        out_padded = self.correction_head(d1)
        return out_padded[:, :, :H, :W]

    def predict_quantiles(self, x, bicubic_raw, coarse_nn_raw):
        raw = self.forward(x)
        p50_correction = raw[:, 0:1, :, :]
        delta_low = F.softplus(raw[:, 1:2, :, :])
        delta_high = F.softplus(raw[:, 2:3, :, :])
        p50_raw = torch.clamp(bicubic_raw + p50_correction, min=0.0)
        p50 = self.mass_conservation(p50_raw, coarse_nn_raw)
        p10 = torch.clamp(p50 - delta_low, min=0.0)
        p90 = p50 + delta_high
        return {"p10": p10, "p50": p50, "p90": p90}

In [ ]:
# --- quantile_loss.py --- verified against pinball loss's asymmetric-penalty property, see DEV_LOG 2026-09-25
HEAVY_THRESHOLD_MM = 64.5
HEAVY_WEIGHT = 4.0
QUANTILES = {"p10": 0.1, "p50": 0.5, "p90": 0.9}


def pinball_loss(pred, truth, tau):
    diff = truth - pred
    return torch.maximum(tau * diff, (tau - 1) * diff)


class QuantileHeavyRainLoss(nn.Module):
    def __init__(self, heavy_threshold=HEAVY_THRESHOLD_MM, heavy_weight=HEAVY_WEIGHT):
        super().__init__()
        self.heavy_threshold = heavy_threshold
        self.heavy_weight = heavy_weight

    def forward(self, preds, truth, mask=None):
        log_truth = torch.log1p(truth)
        weight = 1.0 + self.heavy_weight * (truth >= self.heavy_threshold).float()
        total = 0.0
        for name, tau in QUANTILES.items():
            log_pred = torch.log1p(preds[name])
            pin = pinball_loss(log_pred, log_truth, tau) * weight
            if mask is not None:
                pin = pin[:, :, mask]
            total = total + pin.mean()
        return total / len(QUANTILES)

## Sanity check the inlined model code before training (cheap, catches a transcription error instantly)

In [ ]:
stats = compute_normalization_stats(terrain, coarse_bicubic[splits["train"]])
_x = torch.from_numpy(build_input_tensor(coarse_bicubic[:4], terrain, stats))
_bicubic = torch.from_numpy(coarse_bicubic[:4]).unsqueeze(1).float()
_coarse_nn = torch.from_numpy(nn_predict(fine[:4])).unsqueeze(1).float()

_model = ResidualUNet(in_channels=_x.shape[1], base_channels=BASE_CHANNELS)
_model.eval()
with torch.no_grad():
    _q = _model.predict_quantiles(_x, _bicubic, _coarse_nn)

assert (_q["p10"] <= _q["p50"]).all() and (_q["p50"] <= _q["p90"]).all(), "Monotonicity broken - do not train, check the inlined code above against ml/model/unet.py"
assert not any(torch.isnan(_q[k]).any() for k in _q), "NaN in a fresh forward pass - do not train"
print("Inlined model code sanity check passed: shapes, monotonicity, and NaN-freedom all confirmed on a real batch.")
del _x, _bicubic, _coarse_nn, _model, _q

## Dataset + DataLoaders

In [ ]:
class GramCastDataset(Dataset):
    def __init__(self, indices, fine, coarse_bicubic, terrain, stats):
        self.indices = indices
        self.fine = fine[indices]
        self.coarse_bicubic = coarse_bicubic[indices]
        self.coarse_nn = nn_predict(self.fine)
        self.input_tensor = build_input_tensor(self.coarse_bicubic, terrain, stats)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return {
            "x": torch.from_numpy(self.input_tensor[i]),
            "bicubic_raw": torch.from_numpy(self.coarse_bicubic[i]).unsqueeze(0).float(),
            "coarse_nn_raw": torch.from_numpy(self.coarse_nn[i]).unsqueeze(0).float(),
            "truth": torch.from_numpy(self.fine[i]).unsqueeze(0).float(),
        }


train_ds = GramCastDataset(splits["train"], fine, coarse_bicubic, terrain, stats)
val_ds = GramCastDataset(splits["val"], fine, coarse_bicubic, terrain, stats)
test_ds = GramCastDataset(splits["test"], fine, coarse_bicubic, terrain, stats)

# num_workers=0: all data is already in memory (loaded via np.load above), so extra worker processes give
# minimal speedup here and add a platform-dependent multiprocessing risk (confirmed locally on Windows - a
# freeze_support error - not worth risking on an unverified Kaggle/Linux run for negligible benefit).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## Training loop — checkpoints every epoch (Kaggle free sessions can disconnect; resume via `RESUME_FROM` in the config cell)

In [ ]:
def run_epoch(model, loader, loss_fn, mask, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        x = batch["x"].to(device)
        bicubic_raw = batch["bicubic_raw"].to(device)
        coarse_nn_raw = batch["coarse_nn_raw"].to(device)
        truth = batch["truth"].to(device)
        with torch.set_grad_enabled(is_train):
            preds = model.predict_quantiles(x, bicubic_raw, coarse_nn_raw)
            loss = loss_fn(preds, truth, mask)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

In [ ]:
mask = torch.from_numpy(valid_mask).to(device)
model = ResidualUNet(in_channels=train_ds.input_tensor.shape[1], base_channels=BASE_CHANNELS).to(device)
loss_fn = QuantileHeavyRainLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

start_epoch = 1
best_val_loss = float("inf")
log_mode = "w"
log_path = LOG_DIR / f"{RUN_NAME}.csv"

if RESUME_FROM is not None:
    print(f"Resuming from {RESUME_FROM}")
    ckpt = torch.load(RESUME_FROM, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", ckpt["val_loss"])
    log_mode = "a"
    print(f"Resumed at epoch {start_epoch}, best_val_loss so far: {best_val_loss:.4f}")

with open(log_path, log_mode, newline="") as f:
    writer = csv.writer(f)
    if log_mode == "w":
        writer.writerow(["epoch", "train_loss", "val_loss", "lr", "batch_size", "base_channels", "seed", "timestamp", "elapsed_s"])

    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        train_loss = run_epoch(model, train_loader, loss_fn, mask, device, optimizer)
        val_loss = run_epoch(model, val_loader, loss_fn, mask, device, optimizer=None)
        elapsed = time.time() - t0

        print(f"[{RUN_NAME}] epoch {epoch}/{EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} ({elapsed:.1f}s)")
        writer.writerow([epoch, train_loss, val_loss, LR, BATCH_SIZE, BASE_CHANNELS, SEED, time.strftime("%Y-%m-%d %H:%M:%S"), f"{elapsed:.1f}"])
        f.flush()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {"epoch": epoch, "model_state": model.state_dict(), "val_loss": val_loss, "stats": stats,
                 "config": {"lr": LR, "batch_size": BATCH_SIZE, "base_channels": BASE_CHANNELS, "seed": SEED}},
                CHECKPOINT_DIR / f"{RUN_NAME}_best.pt",
            )

        torch.save(
            {"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
             "val_loss": val_loss, "best_val_loss": best_val_loss, "stats": stats,
             "config": {"lr": LR, "batch_size": BATCH_SIZE, "base_channels": BASE_CHANNELS, "seed": SEED}},
            CHECKPOINT_DIR / f"{RUN_NAME}_last.pt",
        )

print(f"Done. Best val_loss: {best_val_loss:.4f}. Log: {log_path}")
print("If this session disconnects mid-run: start a new session, set RESUME_FROM to the *_last.pt checkpoint")
print("(download it from the Output tab first, or re-add this notebook's own output as a Dataset input), and re-run from the config cell.")

## Next steps after this run finishes

This notebook trains and checkpoints. Once real numbers exist, download `*_best.pt` from the Output tab and, back in the local repo:
1. Run the full evaluation (RMSE/MAE/corr, POD/FAR/CSI, P99 bias, plus coverage checks on the P10/P90 band) against the test set using `ml/baselines/metrics.py`.
2. Compare honestly against the 3 baselines already in `ml/baselines/compare.py` — report the real result even if the U-Net doesn't clearly win on every metric, per this project's own standing rule.
3. Independent station validation and split-conformal calibration of the P10/P90 band are still open ML issues — see `docs/ISSUES_PLAN.md`.

None of these numbers should be written up or reported until they come from an actual run of this notebook — see DEV_LOG.md's standing rule against fabricating results.